# Prediksi Harga Komoditas Pangan DKI Jakarta — XGBoost vs LightGBM

Notebook ini berisi alur lengkap: **training (Optuna + TimeSeriesSplit)** → **evaluasi** → **prediksi harian**.

**Cara pakai**
- **Pertama kali / latih ulang:** jalankan semua sel dari atas (Bagian A → B → C → D).
- **Prediksi harian (model sudah ada di folder `models/`):** cukup jalankan **Bagian A** lalu langsung ke **Bagian D**. Model tidak dilatih ulang, hanya data Excel terbarunya yang dipakai.
- Cocok dijalankan di **Google Colab** (LightGBM tidak terkena blokir Windows) maupun Jupyter lokal.

**Perubahan sesuai masukan dosen**
1. Hyperparameter dicari otomatis dengan **Optuna** + **TimeSeriesSplit** (hanya data latih), bukan hardcode.
2. Model & input window terbaik dipilih dari skor **CV** (data latih), bukan dari data uji.
3. Metrik berbasis rasio ditambahkan: **MAPE** dan **NRMSE** (normalisasi terhadap level harga).
4. **R² = N/A** bila harga aktual periode uji konstan (variansi 0), bukan 0.
5. Grafik memakai **riwayat sejak 2023** + perpanjangan garis waktu prediksi, plus panel zoom.
6. **Evaluasi rekursif multi-hari** (1 hari, 1 minggu, 1 bulan) + pemeriksaan otomatis (fitur prediksi = fitur training, tanpa kebocoran data).
7. Waktu prediksi bisa ditentukan lewat **jumlah hari** atau **tanggal target**.

## Bagian A — Persiapan & fungsi

In [ ]:
# Install paket (di Colab sebagian besar sudah ada). Hapus tanda # jika perlu.
# !pip -q install optuna xgboost lightgbm openpyxl

In [ ]:
import json
import os
import shutil
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

try:  # LightGBM opsional (di Windows dengan kebijakan keamanan ketat DLL-nya bisa diblokir)
    from lightgbm import LGBMRegressor
except (ImportError, OSError) as e:
    LGBMRegressor = None
    print("LightGBM tidak tersedia:", e)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 200)

# ---------------- KONFIGURASI (semua angka eksperimen di sini) ----------------
DATA_FILE = "dataset_harga_preprocessed.xlsx"
MODELS_DIR = "models"
TRAIN_END = "2026-07-31"        # data latih: awal dataset s.d. tanggal ini
TEST_START = "2026-08-01"       # data uji: tanggal ini s.d. akhir dataset
MIN_TEST_ROWS = 10              # jika data uji lebih sedikit dari ini, pakai split 80:20
WINDOW_SIZES = [3, 5, 10, 22]   # 3 hari, 1 minggu, 2 minggu, 1 bulan (hari kerja)
CV_SPLITS = 3                   # jumlah lipatan TimeSeriesSplit
N_TRIALS = 40                   # percobaan Optuna per (atribut, model, window); 15 = cepat, 80 = teliti
SEED = 42
EVAL_HORIZONS = [1, 5, 22]       # evaluasi prediksi rekursif: 1 hari, 1 minggu, 1 bulan kerja
MODEL_NAMES = ["XGBoost", "LightGBM"] if LGBMRegressor is not None else ["XGBoost"]
BASELINE_PARAMS = {"n_estimators": 100, "learning_rate": 0.05}  # pengaturan lama, dicoba pertama sebagai pembanding
print("Model yang dipakai:", MODEL_NAMES)

In [ ]:
# Siapkan file data. Di Colab: upload dataset_harga_preprocessed.xlsx jika belum ada.
if not os.path.exists(DATA_FILE) and not os.path.exists(os.path.join("data", DATA_FILE)):
    try:
        from google.colab import files
        print("Pilih file dataset_harga_preprocessed.xlsx ...")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Taruh '{DATA_FILE}' di folder yang sama dengan notebook ini.")


def find_dataset_path():
    for p in (os.path.join("data", DATA_FILE), DATA_FILE):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"'{DATA_FILE}' tidak ditemukan.")

In [ ]:
ATTR_TO_GROUP = {
    "Beras Kualitas Bawah I": "Beras", "Beras Kualitas Bawah II": "Beras",
    "Beras Kualitas Medium I": "Beras", "Beras Kualitas Medium II": "Beras",
    "Beras Kualitas Super I": "Beras", "Beras Kualitas Super II": "Beras",
    "Daging Ayam Ras Segar": "Daging Ayam",
    "Daging Sapi Kualitas 1": "Daging Sapi", "Daging Sapi Kualitas 2": "Daging Sapi",
    "Telur Ayam Ras Segar": "Telur Ayam",
    "Bawang Merah Ukuran Sedang": "Bawang Merah",
    "Bawang Putih Ukuran Sedang": "Bawang Putih",
    "Cabai Merah Besar": "Cabai Merah", "Cabai Merah Keriting": "Cabai Merah",
    "Cabai Rawit Hijau": "Cabai Rawit", "Cabai Rawit Merah": "Cabai Rawit",
    "Minyak Goreng Kemasan Bermerk 1": "Minyak Goreng", "Minyak Goreng Kemasan Bermerk 2": "Minyak Goreng",
    "Gula Pasir Kualitas Premium": "Gula Pasir", "Gula Pasir Lokal": "Gula Pasir",
}


def _read_raw(path=None):
    df = pd.read_excel(path or find_dataset_path())
    if "Tanggal" in df.columns:
        df["Tanggal"] = pd.to_datetime(df["Tanggal"])
        df = df.set_index("Tanggal")
    elif df.index.name != "Tanggal":
        df.index = pd.to_datetime(df.index)
        df.index.name = "Tanggal"
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.sort_index()


def load_and_preprocess_data(path=None):
    """Baca Excel -> tabel harga rapi (urut tanggal, sel kosong diisi ffill/bfill/interpolasi)."""
    return _read_raw(path).ffill().bfill().interpolate(method="linear")


def check_data_quality(path=None, max_stale_days=4):
    """Cek cepat setelah data diupdate manual. Mengembalikan daftar peringatan (kosong = aman)."""
    msgs = []
    raw = _read_raw(path)
    if raw.index.duplicated().any():
        dup = raw.index[raw.index.duplicated()].strftime("%Y-%m-%d").tolist()
        msgs.append(f"Ada tanggal ganda: {', '.join(dup[:5])}. Hapus salah satunya.")
    last_date = raw.index.max()
    stale = (pd.Timestamp.today().normalize() - last_date).days
    if stale > max_stale_days:
        msgs.append(f"Data terakhir {last_date:%d %b %Y} ({stale} hari lalu). Tambahkan harga terbaru ke Excel.")
    empty = raw.columns[raw.tail(1).isna().any()].tolist()
    if empty:
        msgs.append(f"Baris terakhir ({last_date:%d %b %Y}) kosong untuk: {', '.join(empty[:4])}"
                    f"{'...' if len(empty) > 4 else ''} (diisi otomatis dengan harga sebelumnya).")
    unknown = [c for c in raw.columns if c not in ATTR_TO_GROUP]
    if unknown:
        msgs.append(f"Kolom tidak dikenali (tidak ada model): {', '.join(unknown)}")
    return msgs


def build_features(df_series, window_size):
    """Fitur lag, statistik bergulir (tanpa kebocoran target), dan fitur kalender."""
    df_feat = pd.DataFrame(index=df_series.index)
    df_feat["target"] = df_series.values
    for l in range(1, window_size + 1):
        df_feat[f"lag_{l}"] = df_series.shift(l)
    # shift(1) agar harga hari target tidak ikut dihitung.
    # std = np.std(ddof=1), BUKAN rolling().std() pandas: algoritma online pandas menghasilkan derau (~1e-4)
    # pada jendela harga konstan, sedangkan saat prediksi hasilnya tepat 0 (ketidakcocokan training vs prediksi).
    shifted = df_series.shift(1)
    df_feat[f"rolling_mean_{window_size}"] = shifted.rolling(window_size).mean()
    df_feat[f"rolling_std_{window_size}"] = shifted.rolling(window_size).apply(lambda x: np.std(x, ddof=1), raw=True)
    df_feat["dayofweek"] = df_feat.index.dayofweek
    df_feat["month"] = df_feat.index.month
    df_feat["quarter"] = df_feat.index.quarter
    return df_feat.dropna()


def make_feature_row(history, date, window_size):
    """Satu baris fitur untuk memprediksi harga pada `date` dari riwayat `history`. HARUS sama dengan build_features()."""
    s = pd.Series(history)
    feat = {f"lag_{l}": s.iloc[-l] for l in range(1, window_size + 1)}
    w = s.iloc[-window_size:]
    feat[f"rolling_mean_{window_size}"] = w.mean()
    feat[f"rolling_std_{window_size}"] = np.std(w.values, ddof=1)
    feat["dayofweek"], feat["month"], feat["quarter"] = date.dayofweek, date.month, date.quarter
    return feat


def predict_future_recursive(model, df_series, steps_ahead, window_size, future_dates=None):
    """Prediksi multi-step rekursif. Default tanggal = hari kerja setelah data terakhir."""
    history = list(df_series.values)
    if future_dates is None:
        future_dates = pd.date_range(start=df_series.index[-1] + pd.Timedelta(days=1),
                                     periods=steps_ahead * 2, freq="B")[:steps_ahead]
    preds = []
    for f_date in future_dates[:steps_ahead]:
        p = model.predict(pd.DataFrame([make_feature_row(history, f_date, window_size)]))[0]
        preds.append(p)
        history.append(p)
    return pd.Series(preds, index=list(future_dates[:steps_ahead]))


def steps_to_date(last_date, target_date):
    """Jumlah hari kerja dari hari setelah `last_date` sampai `target_date` (akhir pekan tidak dihitung)."""
    start = (pd.Timestamp(last_date) + pd.Timedelta(days=1)).date()
    end = (pd.Timestamp(target_date) + pd.Timedelta(days=1)).date()
    return int(np.busday_count(start, end))


def check_feature_parity(series, window_size, n_checks=5, rtol=1e-9):
    """Uji kebenaran: fitur saat PREDIKSI harus identik dengan fitur saat TRAINING pada tanggal yang sama."""
    df_feat = build_features(series, window_size)
    for date in df_feat.index[-n_checks:]:
        pos = series.index.get_loc(date)
        row = make_feature_row(series.iloc[:pos].values, date, window_size)
        expected = df_feat.loc[date].drop("target")
        got = np.array([row[k] for k in expected.index], dtype=float)
        if not np.allclose(got, expected.values.astype(float), rtol=rtol, atol=1e-9):
            raise ValueError(f"Fitur prediksi != fitur training pada {date:%Y-%m-%d} (window {window_size}).")
    return True


def load_model(attr):
    """Muat model hasil training (tanpa latih ulang). Mengembalikan (model, info)."""
    safe = attr.replace(" ", "_")
    mp, ip = os.path.join(MODELS_DIR, f"{safe}.joblib"), os.path.join(MODELS_DIR, f"{safe}_info.joblib")
    if not (os.path.exists(mp) and os.path.exists(ip)):
        raise FileNotFoundError(f"Model '{attr}' belum ada. Jalankan Bagian B (training) dulu.")
    info = joblib.load(ip)
    try:
        return joblib.load(mp), info
    except OSError as e:
        raise RuntimeError(f"Model {info.get('model_name')} '{attr}' gagal dimuat ({e}). "
                           f"Latih ulang hanya dengan XGBoost (MODEL_NAMES = ['XGBoost']).") from e


def tight_ylim(ax, *arrays):
    """Sumbu-Y mengikuti rentang data yang tampil agar selisih kecil tetap terlihat."""
    vals = np.concatenate([np.asarray(a, dtype=float) for a in arrays if len(a)])
    lo, hi = vals.min(), vals.max()
    pad = max((hi - lo) * 0.15, abs(vals.mean()) * 0.01)
    ax.set_ylim(lo - pad, hi + pad)


def rp_axis(ax):
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))

## Bagian B — Training (Optuna + TimeSeriesSplit)

Untuk setiap komoditas dicoba kombinasi **model × input window**. Hyperparameter tiap kombinasi dicari Optuna
dengan skor **rata-rata MAPE** pada lipatan validasi `TimeSeriesSplit` (hanya data latih). Kombinasi terbaik dipilih dari
skor CV tersebut; data uji hanya dipakai untuk melaporkan hasil.

In [ ]:
# ---------------- Metrik ----------------
def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred) / np.abs(y_true)) * 100)


def nrmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(np.abs(y_true)) * 100)


def safe_r2(y_true, y_pred):
    """R2 tidak terdefinisi bila data aktual konstan (variansi 0). sklearn memberi 0.0 -> kita ganti NaN."""
    if np.var(np.asarray(y_true, dtype=float)) == 0:
        return float("nan")
    return float(r2_score(y_true, y_pred))


def evaluate(y_true, y_pred):
    return {"MAE": float(mean_absolute_error(y_true, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAPE (%)": mape(y_true, y_pred), "NRMSE (%)": nrmse(y_true, y_pred),
            "R2": safe_r2(y_true, y_pred)}


# ---------------- Split data ----------------
def split_train_test(df_feat):
    train_mask, test_mask = df_feat.index <= TRAIN_END, df_feat.index >= TEST_START
    if test_mask.sum() < MIN_TEST_ROWS or train_mask.sum() == 0:      # fallback 80:20
        i = int(len(df_feat) * 0.8)
        train, test = df_feat.iloc[:i], df_feat.iloc[i:]
    else:
        train, test = df_feat.loc[train_mask], df_feat.loc[test_mask]
    return (train.drop(columns=["target"]), train["target"], test.drop(columns=["target"]), test["target"])


# ---------------- Model & Optuna ----------------
def make_model(name, params):
    if name == "XGBoost":
        return XGBRegressor(random_state=SEED, n_jobs=1, verbosity=0, **params)
    return LGBMRegressor(random_state=SEED, n_jobs=1, verbose=-1, subsample_freq=1, **params)


def suggest_params(trial, name):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
    }
    if name == "XGBoost":
        params["min_child_weight"] = trial.suggest_int("min_child_weight", 1, 10)
    else:
        params["num_leaves"] = trial.suggest_int("num_leaves", 4, 64)
        params["min_child_samples"] = trial.suggest_int("min_child_samples", 5, 50)
    return params


def tune_model(name, X_train, y_train, n_trials):
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

    def objective(trial):
        params = suggest_params(trial, name)
        scores = []
        for tr, va in tscv.split(X_train):
            m = make_model(name, params)
            m.fit(X_train.iloc[tr], y_train.iloc[tr])
            scores.append(mape(y_train.iloc[va], m.predict(X_train.iloc[va])))
        return float(np.mean(scores))

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.enqueue_trial(BASELINE_PARAMS)
    study.optimize(objective, n_trials=n_trials)
    return study.best_params, study.best_value


def evaluate_horizons(model, series, window_size, first_test_date):
    """
    Evaluasi jujur prediksi ke depan: dari tiap hari asal di periode uji, model meramal H hari SECARA REKURSIF
    (hanya memakai harga sampai hari asal), lalu dibandingkan dengan harga aktual. Naive = harga hari asal.
    `model` harus dilatih hanya pada data latih.
    """
    max_h, n = max(EVAL_HORIZONS), len(series)
    first_origin = max(series.index.get_loc(first_test_date) - 1, 0)
    pairs = {h: [] for h in EVAL_HORIZONS}
    for p in range(first_origin, n - 1):
        steps = min(max_h, n - 1 - p)
        fut = predict_future_recursive(model, series.iloc[: p + 1], steps, window_size,
                                       future_dates=list(series.index[p + 1: p + 1 + steps]))
        for h in EVAL_HORIZONS:
            if h <= steps:
                pairs[h].append((series.iloc[p + h], fut.iloc[h - 1], series.iloc[p]))
    out = {}
    for h, rws in pairs.items():
        if not rws:
            continue
        a, f, nv = (np.array(x, dtype=float) for x in zip(*rws))
        out[h] = {"N": len(a), "MAE": float(np.mean(np.abs(a - f))), "RMSE": float(np.sqrt(np.mean((a - f) ** 2))),
                  "MAPE (%)": mape(a, f), "NRMSE (%)": nrmse(a, f),
                  "MAE Naive": float(np.mean(np.abs(a - nv))), "MAPE Naive (%)": mape(a, nv)}
    return out


def process_attribute(attr, series, n_trials, model_names):
    group_name = ATTR_TO_GROUP.get(attr, "Lainnya")
    rows, cands = [], []
    for w in WINDOW_SIZES:
        df_feat = build_features(series, window_size=w)
        X_train, y_train, X_test, y_test = split_train_test(df_feat)
        mape_naive = mape(y_test, X_test["lag_1"])          # pembanding: harga besok = harga kemarin
        for name in model_names:
            best_params, cv_mape = tune_model(name, X_train, y_train, n_trials)
            model = make_model(name, best_params).fit(X_train, y_train)   # model evaluasi: hanya data latih
            pred = model.predict(X_test)
            metrics = evaluate(y_test, pred)
            rows.append({"Kelompok": group_name, "Atribut": attr, "Model": name, "Input Window (Hari)": w,
                         "CV MAPE (%)": cv_mape, **metrics, "MAPE Naive (%)": mape_naive, "Dipilih": False})
            cands.append({"row": len(rows) - 1, "name": name, "window": w, "params": best_params,
                          "cv": cv_mape, "m": metrics, "y_test": y_test, "pred": pred,
                          "model": model, "n_train": len(X_train), "train_last": X_train.index.max()})

    best = min(cands, key=lambda c: (round(c["cv"], 10), c["window"], model_names.index(c["name"])))
    rows[best["row"]]["Dipilih"] = True

    # --- Pemeriksaan kebenaran otomatis ---
    check_feature_parity(series, best["window"])                         # fitur prediksi == fitur training
    leak_ok = bool(best["y_test"].index.min() > best["train_last"])      # data uji sesudah data latih
    if not leak_ok:
        raise ValueError(f"{attr}: data uji tidak berada setelah data latih (kebocoran).")
    hz = evaluate_horizons(best["model"], series, best["window"], best["y_test"].index.min())
    horizon_rows = [{"Kelompok": group_name, "Atribut": attr, "Model": best["name"],
                     "Input Window (Hari)": best["window"], "Horizon (Hari Kerja)": h, **m} for h, m in hz.items()]
    check = {"Atribut": attr, "Fitur prediksi = fitur training": True, "Data uji setelah data latih": leak_ok,
             "Baris latih": best["n_train"], "Baris uji": int(len(best["y_test"])),
             "Latih s.d.": str(best["train_last"].date()), "Uji mulai": str(best["y_test"].index.min().date())}

    # Model FINAL untuk prediksi masa depan: parameter terbaik, dilatih dengan seluruh data
    df_feat = build_features(series, window_size=best["window"])
    final_model = make_model(best["name"], best["params"]).fit(df_feat.drop(columns=["target"]), df_feat["target"])
    info = {"model_name": best["name"], "window": best["window"], "params": best["params"], "cv_mape": best["cv"],
            "mae": best["m"]["MAE"], "rmse": best["m"]["RMSE"], "mape": best["m"]["MAPE (%)"],
            "nrmse": best["m"]["NRMSE (%)"], "r2": best["m"]["R2"]}
    test_pred = pd.DataFrame({"Tanggal": best["y_test"].index, "Atribut": attr, "Model": best["name"],
                              "Aktual": best["y_test"].values, "Prediksi": best["pred"]})
    return attr, rows, final_model, info, test_pred, horizon_rows, check

In [ ]:
# ---------------- Jalankan training semua atribut (paralel) ----------------
os.makedirs(MODELS_DIR, exist_ok=True)
df_data = load_and_preprocess_data()
print(f"Atribut: {df_data.shape[1]} | Data: {df_data.index.min():%d %b %Y} - {df_data.index.max():%d %b %Y} | "
      f"Model: {MODEL_NAMES} | Window: {WINDOW_SIZES} | Percobaan Optuna: {N_TRIALS}")

results = Parallel(n_jobs=-1, verbose=5)(
    delayed(process_attribute)(attr, df_data[attr], N_TRIALS, MODEL_NAMES) for attr in df_data.columns)

all_rows, all_test_pred, best_params_dump, all_horizon, all_checks = [], [], {}, [], []
for attr, rows, final_model, info, test_pred, horizon_rows, check in results:
    safe = attr.replace(" ", "_")
    joblib.dump(final_model, os.path.join(MODELS_DIR, f"{safe}.joblib"))
    joblib.dump(info, os.path.join(MODELS_DIR, f"{safe}_info.joblib"))
    all_rows.extend(rows)
    all_test_pred.append(test_pred)
    all_horizon.extend(horizon_rows)
    all_checks.append(check)
    best_params_dump[attr] = {k: info[k] for k in ("model_name", "window", "params", "cv_mape")}

df_eval = pd.DataFrame(all_rows)
df_test_pred = pd.concat(all_test_pred)
df_eval.to_csv("eval_results.csv", index=False)
df_test_pred.to_csv("test_predictions.csv", index=False)
df_horizon = pd.DataFrame(all_horizon)
df_horizon.to_csv("eval_horizon.csv", index=False)
with open("validation_report.json", "w", encoding="utf-8") as f:
    json.dump(all_checks, f, indent=2, ensure_ascii=False)
with open("best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params_dump, f, indent=2, ensure_ascii=False)
print("Pemeriksaan otomatis (fitur prediksi = fitur training, tanpa kebocoran data): LULUS untuk", len(all_checks), "atribut")
print("Training selesai. Tersimpan: models/, eval_results.csv, eval_horizon.csv, test_predictions.csv, validation_report.json, best_params.json")

## Bagian C — Evaluasi

- **MAPE / NRMSE:** metrik rasio (normalisasi terhadap level harga) sehingga adil antar komoditas.
- **R² = N/A:** harga aktual periode uji konstan (variansi 0), sehingga R² tidak terdefinisi.
- **MAPE Naive:** galat bila harga besok dianggap sama dengan kemarin (pembanding).

In [ ]:
FMT = {"CV MAPE (%)": "{:.2f}", "MAE": "{:,.2f}", "RMSE": "{:,.2f}", "MAPE (%)": "{:.2f}",
       "NRMSE (%)": "{:.2f}", "R2": "{:.4f}", "MAPE Naive (%)": "{:.2f}"}

def show(df):
    display(df.style.format({k: v for k, v in FMT.items() if k in df.columns}, na_rep="N/A"))

# 1) Ringkasan rata-rata per model x window
metric_cols = ["CV MAPE (%)", "MAE", "RMSE", "MAPE (%)", "NRMSE (%)", "R2"]
summary = df_eval.groupby(["Model", "Input Window (Hari)"])[metric_cols].mean().reset_index()
print("Ringkasan rata-rata (R2 hanya dari atribut yang terdefinisi):")
show(summary)

n_na = int(df_eval.groupby("Atribut")["R2"].apply(lambda s: s.isna().all()).sum())
print(f"\nR2 tidak terdefinisi (data uji konstan) pada {n_na} dari {df_eval['Atribut'].nunique()} atribut.")

best_row = summary.loc[summary["CV MAPE (%)"].idxmin()]
print(f"Kombinasi dengan rata-rata CV MAPE terendah: {best_row['Model']}, window {int(best_row['Input Window (Hari)'])} hari")

In [ ]:
# 2) Model & window terbaik per atribut (dipilih dari CV; kolom MAE-R2 = hasil data uji)
best_df = df_eval[df_eval["Dipilih"]].sort_values(["Kelompok", "Atribut"]).drop(columns=["Dipilih"])
show(best_df.reset_index(drop=True))

In [ ]:
# 3) Pengaruh input window terhadap MAPE
pivot = df_eval.pivot_table(index="Atribut", columns="Input Window (Hari)", values="MAPE (%)", aggfunc="mean")
ax = pivot.plot(kind="bar", figsize=(11, 5), width=0.8)
ax.set_ylabel("MAPE (%) (rendah = lebih baik)")
ax.set_title("Pengaruh Input Window Terhadap Kesalahan Prediksi Relatif (MAPE)")
plt.xticks(rotation=45, ha="right"); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# 3b) Evaluasi prediksi REKURSIF multi-hari (cara kerja prediksi sebenarnya): 1 hari, 1 minggu, 1 bulan kerja
hz_sum = df_horizon.groupby("Horizon (Hari Kerja)").agg(
    MAPE_Model=("MAPE (%)", "mean"), MAPE_Naive=("MAPE Naive (%)", "mean"), NRMSE_Model=("NRMSE (%)", "mean"),
    Model_lebih_baik_dari_Naive=("MAPE (%)", lambda s: int((s.values < df_horizon.loc[s.index, "MAPE Naive (%)"].values).sum())))
display(hz_sum.style.format({"MAPE_Model": "{:.2f}", "MAPE_Naive": "{:.2f}", "NRMSE_Model": "{:.2f}"}))
print("Naive = harga dianggap tetap seperti hari asal. Jika MAPE Model > MAPE Naive, model belum memberi nilai tambah "
      "pada horizon itu. Pada harga konstan di periode uji, Naive tidak bisa dikalahkan (galat 0).")
ax = hz_sum[["MAPE_Model", "MAPE_Naive"]].plot(kind="bar", figsize=(7, 4))
ax.set_ylabel("MAPE rata-rata (%)"); ax.set_title("Galat Prediksi Rekursif per Horizon"); ax.grid(True, alpha=0.3)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# 4) Grafik aktual vs prediksi periode uji, dengan riwayat sejak 2023
def plot_eval(attr):
    tp = df_test_pred[df_test_pred["Atribut"] == attr].set_index("Tanggal")
    full = df_data[attr]
    r = best_df[best_df["Atribut"] == attr].iloc[0]
    r2txt = "N/A (data uji konstan)" if pd.isna(r["R2"]) else f"{r['R2']:.4f}"
    print(f"{attr} -> {r['Model']}, window {int(r['Input Window (Hari)'])} | MAE Rp {r['MAE']:,.0f} | "
          f"MAPE {r['MAPE (%)']:.2f}% | NRMSE {r['NRMSE (%)']:.2f}% | R2 {r2txt}")

    fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(11, 11), gridspec_kw={"height_ratios": [2, 2, 1]})
    a1.plot(full.index, full.values, color="black", lw=1.2, label="Aktual (seluruh riwayat)")
    a1.plot(tp.index, tp["Prediksi"], color="red", lw=1.6, label="Prediksi (periode uji)")
    a1.axvspan(tp.index.min(), tp.index.max(), color="orange", alpha=0.12, label="Periode uji")
    a1.set_title(f"Riwayat Harga {attr} sejak {full.index[0]:%b %Y}", fontweight="bold"); a1.legend(fontsize=8)

    a2.plot(tp.index, tp["Aktual"], color="black", marker="o", ms=3, lw=1.4, label="Aktual")
    a2.plot(tp.index, tp["Prediksi"], color="red", marker="x", ms=4, ls="--", lw=1.4, label="Prediksi")
    tight_ylim(a2, tp["Aktual"].values, tp["Prediksi"].values)
    a2.set_title("Zoom periode uji (sumbu-Y disesuaikan)", fontweight="bold"); a2.legend(fontsize=8)

    err = (tp["Prediksi"] - tp["Aktual"]) / tp["Aktual"] * 100
    a3.bar(tp.index, err, color=np.where(err >= 0, "#d62728", "#1f77b4"), width=0.8)
    a3.axhline(0, color="black", lw=0.8)
    a3.set_title("Galat relatif harian (Prediksi - Aktual) / Aktual", fontweight="bold"); a3.set_ylabel("Galat (%)")
    for a in (a1, a2):
        a.set_ylabel("Harga (Rp)"); rp_axis(a)
    for a in (a1, a2, a3):
        a.grid(True, alpha=0.3)
    fig.autofmt_xdate(); plt.tight_layout(); plt.show()

plot_eval("Cabai Rawit Merah")      # ganti dengan nama atribut lain

In [ ]:
# (opsional) tampilkan hyperparameter terbaik hasil Optuna untuk satu atribut
print(json.dumps(best_params_dump["Cabai Rawit Merah"], indent=2))

In [ ]:
# Simpan hasil ke satu file zip (di Colab otomatis terunduh) agar bisa dipakai di laptop / aplikasi Streamlit
shutil.make_archive("hasil_training", "zip", ".", MODELS_DIR)   # zip folder models/
print("Dibuat: hasil_training.zip (isi folder models/); CSV & JSON ada di folder kerja.")
try:
    from google.colab import files
    files.download("hasil_training.zip"); files.download("eval_results.csv")
    files.download("test_predictions.csv"); files.download("best_params.json")
    files.download("eval_horizon.csv"); files.download("validation_report.json")
except ImportError:
    pass

## Bagian D — Prediksi harian (tanpa latih ulang)

**Alur setiap hari**
1. Tambahkan harga hari ini di baris paling bawah `dataset_harga_preprocessed.xlsx` (di Colab: upload ulang file terbaru, sel di bawah menimpa file lama).
2. Atur `HORIZON_HARI` dan `ATRIBUT_DIPILIH`, lalu jalankan sel-sel di bagian ini.

Model dimuat dari folder `models/`; hanya data terbaru yang dipakai sebagai masukan. Latih ulang (Bagian B) cukup berkala,
mis. tiap beberapa minggu.

In [ ]:
# ---------------- PENGATURAN PREDIKSI ----------------
HORIZON_HARI = 5               # 1 = besok, 5 = 1 minggu kerja, 22 = 1 bulan kerja, atau angka bebas
ATRIBUT_DIPILIH = None         # None = semua komoditas, atau mis. "Cabai Merah Besar"
TANGGAL_TARGET = None          # atau mis. "2026-11-06" -> HORIZON_HARI dihitung otomatis sampai tanggal itu
UPLOAD_DATA_BARU = False       # True (di Colab) = upload ulang Excel yang sudah diupdate

if UPLOAD_DATA_BARU:
    from google.colab import files
    if os.path.exists(DATA_FILE):
        os.remove(DATA_FILE)
    files.upload()

In [ ]:
df_now = load_and_preprocess_data()          # data terbaru dari Excel
if TANGGAL_TARGET:
    HORIZON_HARI = steps_to_date(df_now.index.max(), TANGGAL_TARGET)
    if HORIZON_HARI < 1:
        raise ValueError("Tanggal target harus hari kerja setelah data terakhir.")
for msg in check_data_quality():
    print("[PERINGATAN]", msg)
print(f"Data terakhir: {df_now.index.max():%d %b %Y} | Horizon: {HORIZON_HARI} hari kerja\n")

attrs = [ATRIBUT_DIPILIH] if ATRIBUT_DIPILIH else [a for a in ATTR_TO_GROUP if a in df_now.columns]
long_rows, ringkas = [], []
for attr in attrs:
    try:
        model, info = load_model(attr)
    except (FileNotFoundError, RuntimeError) as err:
        print(f"[LEWAT] {attr}: {err}")
        continue
    future = predict_future_recursive(model, df_now[attr], HORIZON_HARI, info["window"])
    last_price = df_now[attr].iloc[-1]
    for tgl, harga in future.items():
        long_rows.append({"Tanggal": tgl.date(), "Atribut": attr, "Model": info["model_name"], "Prediksi": harga})
    ringkas.append({"Atribut": attr, "Model": f"{info['model_name']} (w={info['window']})",
                    "Harga Terakhir": last_price, f"Prediksi {future.index[-1]:%d %b}": future.iloc[-1],
                    "Perubahan (%)": (future.iloc[-1] - last_price) / last_price * 100})

df_ringkas = pd.DataFrame(ringkas)
df_hasil = pd.DataFrame(long_rows)
display(df_ringkas.style.format({c: "{:,.2f}" for c in df_ringkas.columns if c not in ("Atribut", "Model")}))
df_hasil.to_csv("hasil_prediksi.csv", index=False)
print("\nDetail per hari tersimpan di hasil_prediksi.csv")

In [ ]:
# Grafik: riwayat sejak 2023 + perpanjangan garis waktu prediksi (atas), dan zoom (bawah)
GRAFIK_ATRIBUT = "Cabai Merah Besar"      # ganti sesuai kebutuhan
ZOOM_HARI = 90

def plot_forecast(attr, horizon=HORIZON_HARI):
    model, info = load_model(attr)
    hist = df_now[attr]
    future = predict_future_recursive(model, hist, horizon, info["window"])
    tp = None
    if os.path.exists("test_predictions.csv"):
        t = pd.read_csv("test_predictions.csv", parse_dates=["Tanggal"])
        tp = t[t["Atribut"] == attr].set_index("Tanggal")

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 8))
    for ax, h in ((a1, hist), (a2, hist.iloc[-ZOOM_HARI:])):
        ax.plot(h.index, h.values, color="black", lw=1.4, label="Data Historis Aktual")
        if tp is not None and len(tp):
            ax.plot(tp.index, tp["Prediksi"], color="#1f77b4", lw=1.3, label="Prediksi pada Periode Uji")
        ax.plot(future.index, future.values, color="red", ls="--", lw=2, marker="o", ms=3,
                label=f"Prediksi Masa Depan ({info['model_name']})")
        ax.axvline(hist.index[-1], color="gray", ls=":", lw=1)
        ax.set_ylabel("Harga (Rp)"); rp_axis(ax); ax.grid(True, alpha=0.3)
    a1.set_title(f"Riwayat Harga sejak {hist.index[0]:%b %Y} + Prediksi: {attr}", fontweight="bold")
    a1.legend(fontsize=8)
    a2.set_title(f"Zoom: {ZOOM_HARI} hari kerja terakhir + prediksi (sumbu-Y disesuaikan)", fontweight="bold")
    arrs = [hist.iloc[-ZOOM_HARI:].values, future.values]
    if tp is not None and len(tp):
        arrs.append(tp.loc[tp.index >= hist.iloc[-ZOOM_HARI:].index[0], "Prediksi"].values)
    tight_ylim(a2, *arrs)
    fig.autofmt_xdate(); plt.tight_layout(); plt.show()
    display(pd.DataFrame({"Tanggal": future.index.strftime("%Y-%m-%d"), "Estimasi Harga (Rp)": future.values})
            .style.format({"Estimasi Harga (Rp)": "Rp {:,.2f}"}))

plot_forecast(GRAFIK_ATRIBUT)